# Prepare a uniform superposition

Adapted from Qualtran v0.7.0, Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from collections import Counter
from IPython.display import SVG, display
from sympy import Symbol
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit

## `PrepareUniformSuperposition`
Prepares a uniform superposition over first $n$ basis states using $O(log(n))$ T-gates.

Performs a single round of amplitude amplification and prepares a uniform superposition over
the first $n$ basis states $|0>, |1>, ..., |n - 1>$. The gate complexity is
$10 * log(L) + 2 * K$ T gates and $2$ single qubit rotation gates, where $n = L * 2^K$.

However, the current T-complexity is $12 * log(L)$ T-gates and $2 + 2 * (K + log(L))$ rotations
because of two open issues:
 - https://github.com/quantumlib/Qualtran/issues/233 and
 - https://github.com/quantumlib/Qualtran/issues/235

#### Parameters
 - `n`: The gate prepares a uniform superposition over first $n$ basis states.
 - `cvs`: Control values for each control qubit. If specified, a controlled version of the gate is constructed. 

#### References
 - [Encoding Electronic Spectra in Quantum Circuits with Linear T Complexity](https://arxiv.org/abs/1805.03662). Fig 12.


In [ ]:
from examples.algorithms.uniform_preparation_catalog import UniformPreparationExample

### Example Instances

In [ ]:
plain = UniformPreparationExample()

In [ ]:
controlled = UniformPreparationExample(controls=(1,))

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Uniform preparation", plain.signature, {}),
    ("Controlled uniform preparation", controlled.signature, {}),
])))

### Call Graph

In [ ]:
for op in (plain, controlled):
    graph = op.call_graph()
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
    display(graph.leaf_counts)

### Append the two five-state preparations

On the active branch, a clean target receives equal amplitudes on the first five labels with the source phase. Borrowed workspace returns to zero. The resource output shows direct instruction counts before synthesis alongside the source-declared model.

In [ ]:
for op in (plain, controlled):
    circuit = DynamicCircuit(op.qubit_count)
    circuit.append(op)
    counts = Counter(gate.name for gate in circuit.to_ir().instructions)
    print("Emitted on", op.qubit_count, "wires", dict(counts))
    print("Source logical resources", op.resource_model())
    assert counts["measure"] == (9 if op.controls else 8)
symbolic = UniformPreparationExample(Symbol("n", positive=True, integer=True))
display(symbolic.resource_model())